# Titanic
Suited for binary logistic regression

In [ ]:
# Necessary libraries for data manipulation and visualization
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Necessary libraries of scikit-learn for machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Download latest version
path = kagglehub.dataset_download("heptapod/titanic")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv('/run/media/rifat/Felicitous/Machine_Learning_Specialization/Logistic_Regression/datasets/train_and_test2.csv')
df.sample(5)

### Exploratory Data Analysis

In [ ]:
print(df.info())

In [ ]:
print(df['2urvived'].value_counts())
print(df['2urvived'].nunique())

In [ ]:
print(df['Sex'].value_counts())
print(df['Sex'].nunique())

In [ ]:
print(df['2urvived'].value_counts().index)
print(df['2urvived'].value_counts().values)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 5))
ax[0].bar(df['2urvived'].value_counts().index, df['2urvived'].value_counts().values, color=['red', 'green'], alpha=0.7)
ax[0].set_xticks(df['2urvived'].value_counts().index)
ax[0].set_xticklabels(['Not Survived', 'Survived'])
ax[0].set_title('Survival Count')
ax[0].set_xlabel('Survived')
ax[0].set_ylabel('Count')


ax[1].hist(df[df['2urvived'] == 0]['Age'].dropna(), bins=20, alpha=0.7, label='Not Survived', color='red')
ax[1].hist(df[df['2urvived'] == 1]['Age'].dropna(), bins=20, alpha=0.7, label='Survived', color='green')
ax[1].legend()
ax[1].set_xlabel('Age')
ax[1].set_ylabel('Count')
ax[1].set_title('Survived vs Age')
ax[1].grid(True, linestyle='--', alpha=0.3)


plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df[df['2urvived'] == 0]['Fare'].dropna(), bins=20, alpha=0.7, label='Not Survived', color='red')
plt.hist(df[df['2urvived'] == 1]['Fare'].dropna(), bins=20, alpha=0.7, label='Survived', color='green')
plt.legend()
plt.xlabel('Fare')
plt.ylabel('Count')
plt.title('Survived vs Fare')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
# Check for columns with only one unique value
zero_cols = [c for c in df.columns if df[c].nunique() == 1]
cols_to_drop = zero_cols + [c for c in df.columns if c == 'Passengerid']

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(df)

In [ ]:
df.rename(columns={'2urvived': 'Survived', 'sibsp': 'siblings_spouses', 'Parch': 'parents_children', 'Pclass': 'Passenger_Class'}, inplace=True)
df.info()

In [ ]:
for c in df.columns:
    print(f"Column: {c}, Unique Values: {df[c].nunique()}")

In [ ]:
correlation = df.corr()['Survived'].sort_values(ascending=False)
print(correlation)

In [ ]:
# Correlation visualization using matplotlib
plt.style.use('dark_background')

plt.figure(figsize=(10, 8))
corr_matrix = df.corr()
plt.imshow(corr_matrix, cmap='seismic', aspect='auto')

plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.colorbar(label='Correlation')
plt.title('Correlation with Survived')
plt.xlabel('Features')
plt.ylabel('Features')
plt.show()

### Train Test Split

In [ ]:

X = df.drop('Survived', axis=1)
y = df['Survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Check for missing values in the features or null
print(X_train.isnull().sum(),'\n')
print((y.value_counts()))

In [ ]:
# Fill missing values in the 'Embarked' column with the mode (most frequent value)

X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])
X_test['Embarked'] = X_test['Embarked'].fillna(X_test['Embarked'].mode()[0])

print(X_train['Embarked'].sample(5))
print(X_test['Embarked'].sample(5))

### Feature Scaling

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

In [ ]:
scores = cross_val_score(pipeline,
                         X_train, 
                         y_train, 
                         cv=5, 
                         scoring='accuracy')

print("CV scores:", scores)
print("Mean CV accuracy:", scores.mean())

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
# Coefficients and intercept of the logistic regression model
print(f"Coefficients: \n{model.coef_}\n\nIntercept: {model.intercept_}")

### Prediction

In [ ]:
# Predicting the target variable for the test set
y_pred = pipeline.predict(X_test)
print(f"Predicted values: {y_pred}")

In [ ]:
# Evaluate the model's performance using accuracy, confusion matrix, and classification report
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}\n")